<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week11_data_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** These exercises are meant to be worked through yourself – treat this the same as switching off a calculator's solver mode in an exam. It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 11 – Data Exercises: Regression with Time Series Data

Adapted from Bekes & Kezdi, *Data Analysis for Business, Economics, and Policy* (Ch12) – the two workshop case studies (stock returns, and Arizona electricity vs. temperature), extended with real data. No output shown here to check yourself against: the point is to practise on data and questions you haven't seen the answer for.

Easier and/or shorter exercises are marked **[\*]**; harder and/or longer exercises are marked **[\*\*]**.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

stocks_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/stocks_monthly.csv"
az_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/arizona_electricity.csv"

stocks = pd.read_csv(stocks_path, parse_dates=['date'])
az = pd.read_csv(az_path, parse_dates=['date'])
stocks.head()

`stocks` now has `p_AAPL`/`pct_aapl` (Apple, built the same way as the workshop's `p_MSFT`/`pct_msft`) and `riskfree_annual_pct`/`riskfree_monthly_pct` (the real 3-month Treasury bill rate from FRED, for Question 2) alongside the workshop's original MSFT/S&P 500 columns.

`az` now also has `prop_days_over_90`/`prop_days_under_70` (the share of days that month above 90°F / below 70°F, from real Phoenix airport station data) alongside the workshop's original `CD`/`HD`/`dCD`/`dHD` columns.

> **Syntax hint – a publication-style comparison table.** Whenever a question below asks you to compare models, present them side by side:
>
> ```python
> def stars(p):
>     return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
>
> def compare_models(models, names, variables):
>     rows = []
>     for param_name, label in variables:
>         row = {'Variable': label}
>         for name, m in zip(names, models):
>             row[name] = f"{m.params[param_name]:.3f}{stars(m.pvalues[param_name])}" if param_name in m.params.index else ''
>         rows.append(row)
>     rows.append({'Variable': 'N', **{name: str(int(m.nobs)) for name, m in zip(names, models)}})
>     rows.append({'Variable': 'R\u00b2', **{name: f'{m.rsquared:.3f}' for name, m in zip(names, models)}})
>     return pd.DataFrame(rows)
> ```

## Question 1 [\*]

Use `stocks` and reproduce the workshop's "returns on a company stock and market returns" case study, but with **Apple** (`pct_aapl`) instead of Microsoft.

1. Regress `pct_aapl` on `pct_sp500`.
2. Compare your beta to the workshop's own MSFT beta (a comparison table with both models side by side works well here).

## Question 2 [\*]

In finance, "beta" is more properly the regression of **excess return** (return above the risk-free rate) on the market's excess return. `riskfree_monthly_pct` (a real Treasury-bill rate, converted from an annual to a monthly rate) is already provided.

1. Calculate the excess return for each month as `return - riskfree_monthly_pct`, for both MSFT and the S&P 500.
2. Regress the excess return of MSFT on the excess return of the index.
3. Compare your results with the workshop's plain (non-excess) MSFT-vs-S&P500 regression. Does using excess returns instead of raw returns change the beta much? Why might that be?

## Question 3 [\*]

Use `az` and investigate whether there's nonlinearity in the association between the workshop's variables (`dlnQ`, `dCD`, `dHD`).

1. Try adding a squared term for `dCD` and/or `dHD` to the workshop's `dlnQ ~ dCD + dHD` model.
2. Discuss your results – is there meaningful evidence of nonlinearity here, or does the straight-line version already do the job?

## Question 4 [\*]

Use `az` and look at `prop_days_over_90` (the proportion of days that month with a high temperature over 90°F) and `prop_days_under_70` (the proportion under 70°F).

1. Estimate a regression analogous to the workshop's case study, but using these two proportions (differenced, to match the workshop's own `dCD`/`dHD` approach: `.diff()`) in place of `dCD` and `dHD`.
2. Interpret your coefficients, and discuss your results.
3. Compare this model to the workshop's original `dCD`/`dHD` model (comparison table, or just compare R²) – which captures more of the variation in electricity consumption, and why might that be? Think about what information a simple day-count proportion throws away that a degree-day total keeps.

## Question 5 [\*]

Use `az` and reproduce the electricity-consumption-and-temperature analysis using **levels** as opposed to **differences** of the variables (`lnQ ~ CD + HD` instead of the workshop's `dlnQ ~ dCD + dHD` – you may already have `lnQ` in the data, or take `np.log(az['Q'])` yourself).

1. Interpret your results.
2. Compare your results (including R²) with the workshop's differenced-variables case study. Does the levels model actually measure the effect of temperature on electricity consumption in a way you'd trust, or could something else be going on?

*Your comparison of the levels model to the differenced model, and which one you'd actually trust:*